# MWOW Hurricane Observation Demo

This notebook demonstrates **MWOW (Multi-sensor Worldwide Ocean Winds)** data access
and visualization tools using 2025 Atlantic hurricane season cases.

## Sensors in MWOW (non-EPI product)

| Sensor | Platform | Type | Resolution |
|--------|----------|------|------------|
| ASCAT-B | MetOp-B | C-band scatterometer | 25 km |
| ASCAT-C | MetOp-C | C-band scatterometer | 25 km |
| EOS-6 (ScatSat-1) | OceanSat-3 | Ku-band scatterometer | 25 km |
| SMAP | SMAP | L-band radiometer | 40 km |
| SWOT | SWOT | Ka-band altimeter | 5 km (native) |

## Target Storms

| Storm | Dates | Peak | Region |
|-------|-------|------|--------|
| Hurricane Erin | Aug 11–22 | Cat 5, 160 mph | Cape Verde → Caribbean → Bermuda |
| Hurricane Humberto | Sep 24–Oct 1 | Cat 5, 160 mph | Leeward Is. → Bermuda |
| Hurricane Melissa | Oct 21–31 | Cat 5, 190 mph | Caribbean → Bahamas → NE US |

## Setup

```bash
conda activate mwow-user-tools
pip install -e /path/to/mwow-user-tools
```

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

from mwow_tools import (
    open_mwow_files, select_region, select_point, match_ship_track,
    generate_track_video, generate_region_video,
    plot_wind_map, plot_sensor_coverage,
    collocate_files, plot_joint_histogram,
    SENSOR_IDS, SENSOR_NAMES,
)
from mwow_tools.video import MWOW_JET_CMAP
import cartopy.crs as ccrs
import cartopy.feature as cfeature

MS_TO_KT = 1.9438  # m/s to knots conversion

# Data root for non-EPI (no Chinese sensors) lowres product
DATA_ROOT = "/u/tsali-z0/fore/mwow_v0.2_fwd/nonepi/lowres"
EPI_ROOT = "/u/tsali-z0/fore/mwow_v0.2_fwd/epi/lowres"

# Track data
TRACK_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")


def get_files(start_date, end_date, root=DATA_ROOT):
    """Gather MWOW file paths for a date range (inclusive).
    
    Parameters
    ----------
    start_date, end_date : str
        Dates as 'YYYY-MM-DD'.
    """
    dates = pd.date_range(start_date, end_date, freq="D")
    paths = []
    for d in dates:
        day_dir = os.path.join(root, f"{d.year:04d}/{d.month:02d}/{d.day:02d}")
        paths.extend(sorted(glob.glob(os.path.join(day_dir, "*.nc"))))
    print(f"Found {len(paths)} files for {start_date} to {end_date}")
    return paths

---
## 1. Storm-Tracking Video: Hurricane Melissa

Generate a time-lapse video where the map center follows Hurricane Melissa
as it crosses the Caribbean and intensifies to Category 5 (Oct 21–31, 2025).

The `generate_track_video()` function:
- Interpolates the storm track to each satellite observation time
- Centers the map on the storm for each frame
- Overlays the past track as the storm moves
- Produces an MP4 with frame timing proportional to real observation gaps

In [ ]:
# Load Melissa's best track
melissa_track = os.path.join(TRACK_DIR, "melissa_2025_track.csv")
melissa_df = pd.read_csv(melissa_track, parse_dates=["time"])
print(f"Track points: {len(melissa_df)}")
print(f"Time range: {melissa_df.time.iloc[0]} to {melissa_df.time.iloc[-1]}")
melissa_df.head()

In [ ]:
# Gather files covering Melissa's lifetime
melissa_files = get_files("2025-10-21", "2025-10-31")

# Generate the storm-tracking video
video_path = generate_track_video(
    melissa_files,
    track=melissa_track,
    region_size=5.0,       # Half-width = 5° → 10° × 10° frame
    output_dir=".",
    output_name="melissa_storm_tracking.mp4",
    speedup=14400,         # 4 hours real time = 1 second video
    fps=10,
    speed_range=(0, 30),   # Adjusted for visible color gradation
    qi_max=2,
    title="Hurricane Melissa (2025)",
    show_track=True,
    track_color="magenta",
    timestamp_date_color="navy",
)
print(f"\nVideo output: {video_path}")

---
## 2. Sensor Coverage Map: Hurricane Erin (6-Hour Panels)

Show which satellite sensor observed each pixel during Hurricane Erin's
peak intensity (Aug 16, 2025) broken into 6-hour windows. This demonstrates
how multiple sensors complement each other to provide dense temporal coverage
around a TC, while avoiding overlap that masks individual sensor contributions.

In [ ]:
# Load 6-hourly files for Aug 16 (4 files = 4 six-hour windows)
day_dir = os.path.join(DATA_ROOT, "2025/08/16")
erin_files = sorted(glob.glob(os.path.join(day_dir, "*.nc")))
print(f"Files for Aug 16: {len(erin_files)}")

# Region centered on Erin's approximate position on Aug 16
erin_region = {"lat_center": 19.5, "lon_center": -58.0,
               "lat_size": 10.0, "lon_size": 12.0}

# 2x2 panel layout: one 6-hour window per panel
fig, axes = plt.subplots(2, 2, figsize=(14, 10),
                         subplot_kw={"projection": ccrs.PlateCarree()})
time_labels = ["00-06 UTC", "06-12 UTC", "12-18 UTC", "18-00 UTC"]

for i, (f, ax, tlabel) in enumerate(zip(erin_files, axes.flatten(), time_labels)):
    ds_window = open_mwow_files([f])
    plot_sensor_coverage(ds_window, region=erin_region, qi_max=2,
                         title=f"Aug 16, 2025  {tlabel}", ax=ax)

fig.suptitle("Non-EPI Sensor Coverage: Hurricane Erin (Aug 16, 2025)",
             fontsize=13, fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("erin_coverage_6hr_nonepi.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. Point Time Series: Hurricane Humberto with Buoy Validation

Extract a wind speed time series at NDBC buoy 41049 (27.5°N, 62.3°W, ~300 nm
SSE of Bermuda) as Hurricane Humberto passes through (Sep 24–Oct 1, 2025).

The buoy data (black X markers) provides ground truth for comparison against
the multi-sensor MWOW satellite observations (colored circles). All speeds
are in knots.

In [ ]:
# NDBC buoy 41049 location
buoy_lat, buoy_lon = 27.505, -62.271

# Load MWOW data at buoy location
humberto_files = get_files("2025-09-24", "2025-10-01")
ds_hum = open_mwow_files(humberto_files)
ds_point = select_point(ds_hum, lat=buoy_lat, lon=buoy_lon)

times = ds_point.time.values
speeds = ds_point.wind_speed.values * MS_TO_KT  # Convert to knots
sensor_ids = ds_point.sensor_id.values

valid = np.isfinite(speeds) & ~np.isnat(times)
t_valid = times[valid]
s_valid = speeds[valid]
sid_valid = sensor_ids[valid]

print(f"MWOW observations at buoy: {valid.sum()} over {len(humberto_files)} files")
print(f"Sensors: {[SENSOR_NAMES[int(s)] for s in np.unique(sid_valid[np.isfinite(sid_valid)])]}")

# Load buoy data
buoy_path = os.path.join(TRACK_DIR, "ndbc_41049_sep24_oct01_2025.csv")
buoy = pd.read_csv(buoy_path, parse_dates=["time"])
buoy_valid = buoy.dropna(subset=["wind_speed_ms"])
print(f"Buoy observations: {len(buoy_valid)}")

In [ ]:
# Plot time series: MWOW colored by sensor + buoy as black X markers
fig, ax = plt.subplots(figsize=(12, 4.5))

sensor_colors = {
    0: "#1f77b4", 1: "#2ca02c", 2: "#ff7f0e",
    5: "#9467bd", 6: "#17becf",
}

# Buoy data (hourly average for clarity)
buoy_hourly = buoy_valid.set_index("time").resample("1h").mean().dropna()
ax.scatter(buoy_hourly.index, buoy_hourly["wind_speed_ms"].values * MS_TO_KT,
           marker="x", c="black", s=25, linewidths=1.2,
           label="NDBC 41049 (buoy)", zorder=3)

# MWOW data colored by sensor
for sid in np.unique(sid_valid[np.isfinite(sid_valid)]).astype(int):
    mask = np.isfinite(sid_valid) & (sid_valid.astype(float) == sid)
    ax.scatter(t_valid[mask], s_valid[mask],
               c=sensor_colors.get(sid, "gray"), s=40,
               label=f"MWOW {SENSOR_NAMES[sid]}", alpha=0.9,
               edgecolors="none", zorder=4)

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Wind Speed [kt]")
ax.set_title(f"MWOW vs NDBC Buoy 41049 ({buoy_lat:.1f}°N, {abs(buoy_lon):.1f}°W)\n"
             f"Hurricane Humberto Passage (Sep 24–Oct 1, 2025)")
ax.legend(loc="upper left", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("humberto_timeseries_v2.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4. Regional Wind Maps: SMAP and EOS-6 at Erin's Peak

Plot individual sensor wind fields around Hurricane Erin at its Category 5
peak on August 16, 2025. Each sensor is shown separately (no compositing)
with exact observation time and maximum wind speed compared to best track.

In [ ]:
# Load Aug 16 data and best track
erin_track = pd.read_csv(os.path.join(TRACK_DIR, "erin_2025_track.csv"), parse_dates=["time"])
peak_files = get_files("2025-08-16", "2025-08-16")
ds_peak = open_mwow_files(peak_files)

# Select region around Erin
region = {"lat_center": 19.8, "lon_center": -59.0, "lat_size": 8.0, "lon_size": 8.0}
ds_region = select_region(ds_peak, region["lat_center"], region["lon_center"],
                          lat_size=region["lat_size"], lon_size=region["lon_size"],
                          drop_empty_orbits=False)

sensor_ids_arr = ds_region.sensor_id.values
lats = ds_region.latitude.values
lons = ds_region.longitude.values

# Plot separate SMAP and EOS-6 fields
fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                         subplot_kw={"projection": ccrs.PlateCarree()})

for ax, (sensor_id, sensor_label) in zip(axes, [(5, "SMAP"), (2, "EOS-6")]):
    # Find orbit with highest max wind for this sensor
    orbits = [i for i in range(len(sensor_ids_arr))
              if np.isfinite(sensor_ids_arr[i]) and int(sensor_ids_arr[i]) == sensor_id]
    best_orb, best_max = None, 0
    for orb in orbits:
        ws = ds_region.wind_speed.values[orb]
        qi = ds_region.quality_indicator.values[orb]
        valid = np.isfinite(ws) & (qi <= 2)
        if valid.sum() > 0:
            mx = np.nanmax(ws[valid])
            if mx > best_max:
                best_max = mx
                best_orb = orb

    ws = ds_region.wind_speed.values[best_orb]
    wd = ds_region.wind_direction.values[best_orb]
    qi = ds_region.quality_indicator.values[best_orb]
    valid = np.isfinite(ws) & (qi <= 2)
    max_speed_kt = np.nanmax(ws[valid]) * MS_TO_KT

    # Get observation time
    orb_times = ds_region.time.values[best_orb]
    valid_times = orb_times[~np.isnat(orb_times)]
    med_time = np.sort(valid_times)[len(valid_times) // 2]
    t_pd = pd.Timestamp(med_time)
    time_str = t_pd.strftime("%Y-%m-%d %H:%M UTC")

    # Best track max near this time
    nearby = erin_track[(erin_track.time >= t_pd - pd.Timedelta("3h")) &
                        (erin_track.time <= t_pd + pd.Timedelta("3h"))]
    best_track_max = nearby["max_wind_kt"].max() if len(nearby) > 0 else np.nan

    # Masked field in knots
    ws_plot = np.where(valid, ws * MS_TO_KT, np.nan)
    wd_plot = np.where(valid, wd, np.nan)
    vmax = max(80, int(np.ceil(max_speed_kt / 10) * 10))

    # Render
    ax.set_extent([lons[0], lons[-1], lats[0], lats[-1]], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=1)
    ax.coastlines(resolution="50m", linewidth=0.8, zorder=3)

    norm = Normalize(vmin=0, vmax=vmax)
    im = ax.pcolormesh(lons, lats, ws_plot, cmap=MWOW_JET_CMAP, norm=norm,
                       shading="nearest", transform=ccrs.PlateCarree(), zorder=2)

    # Wind arrows
    sub = max(1, len(lons) // 12)
    lon_sub, lat_sub = lons[::sub], lats[::sub]
    lon_mesh, lat_mesh = np.meshgrid(lon_sub, lat_sub)
    dir_sub = wd_plot[::sub, ::sub]
    spd_sub = ws_plot[::sub, ::sub]
    u = np.sin(np.deg2rad(dir_sub))
    v = np.cos(np.deg2rad(dir_sub))
    arrow_mask = np.isfinite(spd_sub) & np.isfinite(dir_sub)
    ax.quiver(lon_mesh, lat_mesh,
              np.where(arrow_mask, u, np.nan),
              np.where(arrow_mask, v, np.nan),
              scale=25, width=0.003, color="white", alpha=0.8,
              transform=ccrs.PlateCarree(), zorder=4)

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray",
                      alpha=0.5, linestyle="--", zorder=5)
    gl.top_labels = False
    gl.right_labels = False

    fig.colorbar(im, ax=ax, label="Wind Speed [kt]", shrink=0.8, pad=0.05)

    title = f"Hurricane Erin — {sensor_label}\n{time_str}\n"
    if not np.isnan(best_track_max):
        title += f"Best Track: {best_track_max:.0f} kt | {sensor_label}: {max_speed_kt:.0f} kt"
    else:
        title += f"{sensor_label} Max: {max_speed_kt:.0f} kt"
    ax.set_title(title, fontsize=10, fontweight="bold")

fig.tight_layout()
fig.savefig("erin_wind_smap_eos6.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Ship Track Matching: NOAA Ship Pisces

Match satellite wind observations to the track of **NOAA Ship Pisces**
(call sign WTDL) off the US East Coast during April 1–8, 2026.

Ship data source: SAMOS (Shipboard Automated Meteorological and
Oceanographic System) via ERDDAP at COAPS/FSU.

The `match_ship_track()` function finds the nearest MWOW observation in
space and time for each position along the vessel's track. Ship anemometer
readings (squares) are compared against satellite observations (circles)
with sensor names labeled.

In [ ]:
# NOAA Ship Pisces (WTDL) — 6-hourly positions, April 2026
# Source: SAMOS/ERDDAP (fsuNoaaShipWTDLnrt), averaged to 6-hourly
# Using EPI data (nonepi has a sensor_id metadata bug for ~20% of orbits)
ship_all = pd.read_csv(os.path.join(TRACK_DIR, "noaa_pisces_apr2026.csv"), parse_dates=["time"])

# Select clean eastward transit segment (Apr 7 06:00 – Apr 8 18:00)
# 7 positions with 0.44–0.54° spacing, no backtracking
ship = ship_all.iloc[25:32].reset_index(drop=True)
ship_lats = ship["latitude"].values
ship_lons = ship["longitude"].values
ship_times = ship["time"].values
ship_winds_kt = ship["wind_speed_ms"].values * MS_TO_KT

print(f"Ship positions: {len(ship)}")
print(f"Time range: {ship.time.iloc[0]} to {ship.time.iloc[-1]}")
print(f"Wind speed range: {ship_winds_kt.min():.1f} – {ship_winds_kt.max():.1f} kt")

# Load EPI MWOW data covering the transit
ship_files = get_files("2026-04-07", "2026-04-08", root=EPI_ROOT)
ds_ship = open_mwow_files(ship_files)

# Match ship positions to nearest MWOW observations
ds_matched = match_ship_track(ds_ship, ship_lats, ship_lons, ship_times)

speeds_kt = ds_matched.wind_speed.values * MS_TO_KT
sids = ds_matched.sensor_id.values

print(f"\nMatched {len(ship)} ship positions to MWOW observations")
print(f"MWOW wind speeds (kt): min={np.nanmin(speeds_kt):.1f}, max={np.nanmax(speeds_kt):.1f}")
print(f"Sensors: {[SENSOR_NAMES.get(int(s), '?') for s in sids if np.isfinite(s)]}")

In [ ]:
# Plot ship track: squares = anemometer, circles = MWOW satellite with sensor labels
cmap = plt.cm.jet
norm = Normalize(vmin=0, vmax=30)

lat_min, lat_max = ship_lats.min() - 0.5, ship_lats.max() + 0.5
lon_min, lon_max = ship_lons.min() - 0.5, ship_lons.max() + 0.5

fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.coastlines(resolution="10m", linewidth=0.8)
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle="--")
gl.top_labels = False
gl.right_labels = False

# Dashed vessel track
ax.plot(ship_lons, ship_lats, "k--", linewidth=1.5, label="Vessel Track",
        transform=ccrs.PlateCarree())

# Ship anemometer as colored squares
ax.scatter(ship_lons, ship_lats, c=ship_winds_kt, cmap=cmap,
           norm=norm, s=160, marker="s", edgecolors="black", linewidths=0.8,
           transform=ccrs.PlateCarree(), zorder=4, label="Ship anemometer")

# MWOW matched as colored circles (offset south for visibility)
lon_offset, lat_offset = 0.0, -0.15
mwow_lons = ds_matched.longitude.values + lon_offset
mwow_lats = ds_matched.latitude.values + lat_offset

sc_mwow = ax.scatter(mwow_lons, mwow_lats,
                     c=speeds_kt, cmap=cmap, norm=norm,
                     s=120, marker="o", edgecolors="black", linewidths=0.5,
                     transform=ccrs.PlateCarree(), zorder=5, label="MWOW satellite")

# Sensor name labels below circles
for i in range(len(sids)):
    if np.isfinite(sids[i]):
        ax.text(mwow_lons[i], mwow_lats[i] - 0.12,
                SENSOR_NAMES.get(int(sids[i]), "?"),
                fontsize=8, fontweight="bold", color="darkblue",
                ha="center", va="top",
                transform=ccrs.PlateCarree(), zorder=6)

# Time labels above ship positions
for i in range(len(ship_times)):
    t_str = pd.Timestamp(ship_times[i]).strftime("%b %d %H:%M")
    ax.text(ship_lons[i], ship_lats[i] + 0.12, t_str,
            fontsize=7, ha="center", va="bottom", color="gray",
            transform=ccrs.PlateCarree())

fig.colorbar(sc_mwow, ax=ax, label="Wind Speed [kt]", shrink=0.7)
ax.set_title("Ship Track Matching: NOAA Ship Pisces (WTDL)\n"
             "Eastward Transit, April 7–8, 2026\n"
             "Squares = ship anemometer, Circles = MWOW satellite", fontsize=10)
ax.legend(loc="lower right", fontsize=9)
fig.savefig("pisces_ship_track.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Sensor Comparison: Tropical Atlantic

Compare EOS-6 winds against the ASCAT reference in the tropical Atlantic
development region during peak hurricane season (September 2025).

The joint histogram shows the density of matched observation pairs
with bias and standard deviation statistics.

In [ ]:
# Use 3 days for a quick comparison demo
comparison_files = get_files("2025-09-15", "2025-09-17")

# Collocate EOS-6 against ASCAT
collocations = collocate_files(
    comparison_files,
    target_sensor="EOS-6",
    ref_sensors=("ASCAT-B", "ASCAT-C"),
    max_dt_minutes=30,
    qi_max=None,  # Include all QI for sensitivity analysis
)

print(f"Collocated pairs: {len(collocations['ref_speed'])}")

In [ ]:
# Joint histogram: EOS-6 vs ASCAT
plot_joint_histogram(
    collocations["ref_speed"],
    collocations["target_speed"],
    target_name="EOS-6",
    ref_name="ASCAT",
    speed_range=(0, 25),
    save_path="eos6_vs_ascat_histogram.png",
    title="EOS-6 vs ASCAT (Sep 15\u201317, 2025)\nTropical Atlantic",
)
plt.show()

---
## 7. CLI Quick-Start

The `mwow-tools` command-line interface provides zero-code access to common
workflows. Here are example commands:

In [ ]:
%%bash
# Point time series at Bermuda during Humberto
mwow-tools timeseries /u/tsali-z0/fore/mwow_v0.2_fwd/nonepi/lowres/2025/09/27/*.nc \
    --lat 32.0 --lon -65.0 --no-plot

In [ ]:
%%bash
# Regional view near Erin's peak position
mwow-tools region /u/tsali-z0/fore/mwow_v0.2_fwd/nonepi/lowres/2025/08/16/*.nc \
    --lat 20 --lon -59 --size 8 -o erin_cli_region.png

---
## Summary

This notebook demonstrated the following `mwow-user-tools` capabilities:

| Capability | Function | Use Case |
|-----------|----------|----------|
| Storm-tracking video | `generate_track_video()` | TC evolution time-lapse |
| Sensor coverage map | `plot_sensor_coverage()` | Multi-sensor sampling visualization |
| Point time series | `select_point()` | TC passage at fixed location |
| Regional wind map | `plot_wind_map()` | Snapshot of wind structure |
| Ship track matching | `match_ship_track()` | Research vessel validation |
| Sensor comparison | `collocate_files()` + `plot_joint_histogram()` | Inter-sensor validation |
| CLI interface | `mwow-tools` | Zero-code quick-look |

### Data Access

MWOW data will be available from PO.DAAC. Install the tools:
```bash
git clone <repo-url>
pip install -e mwow-user-tools
```

All functions accept glob patterns or lists of file paths, supporting
flexible date range selection across the daily file structure.